In [ ]:
from polyconv_method import (fermion_logZ_numeric, logaddexp,
                             degeneracy_box, level_degeneracy,
                             log_binom, convolve_shell_logspace)

In [ ]:
import numpy as np

# ==========================================
# USER INPUTS: Define your potential here
# ==========================================

# Integration Domain
x_min = -1.0
x_max = 1.0
num_grid = 800

def user_potential(x):
    """
    Define the separable 1D potential V(x).
    Must be able to accept a NumPy array x and return an array of the same shape.
    """
    # --- Example 1: Hard Wall (Infinite Square Well) ---
    # (V=0 inside domain, boundary is handled by x_min, x_max)
    return np.zeros_like(x)

    # --- Example 2: Harmonic Oscillator ---
    # w = 1.0
    # return 0.5 * (w**2) * (x**2)

    # --- Example 3: Double Well ---
    # return (x**2 - 1.0)**2

    # Example 4: Periodic potential
    # V0 = 5.0
    # k = 2.0 * np.pi
    # return V0 * np.sin(k * x)**2

    # Example 5: Anharmonic Oscillator
    # w = 1.0
    # lambda_param = 0.5
    # return 0.5 * (w**2) * (x**2) + lambda_param * (x**4)

    # Example 6: Linear Potential
    # force = 2.0
    # # Combine with a hard wall at x=0
    # V = force * x
    # V[x < 0] = np.inf
    # return V



# Parameters for testing
n = 5
d = 2
N = [2]


In [ ]:
import numpy as np
import scipy.linalg as la

def get_1d_log_eigenvalues(tau, N, x_min, x_max, num_grid):
    epsilon = tau / N
    x = np.linspace(x_min, x_max, num_grid)
    dx = (x_max - x_min) / (num_grid - 1)
    
    X, Y = np.meshgrid(x, x)
    
    # Calculate Potential Energy components
    Vx = user_potential(X)
    Vy = user_potential(Y)
    
    # Calculate Kinetic Matrix
    K_kinetic = (1.0 / np.sqrt(2 * np.pi * epsilon)) * np.exp(-((X - Y)**2) / (2 * epsilon))
    
    # Symmetric Trotter Breakup for Primitive Approximation
    T = np.exp(-epsilon * Vx / 2.0) * K_kinetic * np.exp(-epsilon * Vy / 2.0) * dx
    
    # Extract eigenvalues of the symmetric transfer matrix
    eigvals = la.eigvalsh(T)
    eigvals = eigvals[eigvals > 1e-15] # filter numerical noise
    return np.log(np.sort(eigvals)[::-1])

def get_nd_log_eigenvalues(tau, N, d, x_min, x_max, num_grid, max_states=1000):
    log_eig1 = get_1d_log_eigenvalues(tau, N, x_min, x_max, num_grid)
    
    if d == 1:
        return log_eig1[:max_states]
    elif d == 2:
        # Sum of logs for products of eigenvalues
        log_eig2 = []
        for i in range(len(log_eig1)):
            for j in range(len(log_eig1)):
                if log_eig1[i] + log_eig1[j] > -100:
                    log_eig2.append(log_eig1[i] + log_eig1[j])
        log_eig2 = np.array(log_eig2)
        log_eig2.sort()
        return log_eig2[::-1][:max_states]
    else:
        raise NotImplementedError("Only d=1 and d=2 are implemented for numerical eigenvalues.")

class NumericalEigenvalueLogW:
    def __init__(self, tau, N, d, x_min, x_max, num_grid, max_states=400):
        self.log_eigvals = get_nd_log_eigenvalues(tau, N, d, x_min, x_max, num_grid, max_states)
        self.N = N
        self.max_shell = len(self.log_eigvals) - 1
        
    def gk_fn(self, k):
        return 1
        
    def logwk_fn(self, k, logb):
        return self.log_eigvals[k] * self.N

def fermion_logZ_numeric_pimc_predict(tau, N, n, d=None, **kwargs):
    # Calculate eigenvalues dynamically for the given tau and N using user_potential
    num_eig = NumericalEigenvalueLogW(tau, N, d, x_min, x_max, num_grid, max_states=max(n*4, 100))
    
    # Remove kwargs that conflict
    kwargs.pop('k_start', None)
    kwargs.pop('max_shell', None)
    kwargs.pop('gk_fn', None)
    kwargs.pop('logwk_fn', None)
    kwargs.pop('Ek_fn', None)
    
    return fermion_logZ_numeric(
        tau=tau, N=N, n=n, d=d, k_start=0, max_shell=num_eig.max_shell,
        gk_fn=num_eig.gk_fn, logwk_fn=num_eig.logwk_fn,
        **kwargs
    )


In [ ]:
from plotting import make_tau_grid, finite_difference_user_sign, plot_logZ_and_fd_multiN

In [ ]:
results = plot_logZ_and_fd_multiN(
    tau_start=1.0,
    tau_end=15.0,
    tau_step=1.0,
    
    n=n,
    d=d,
    N_list=N,
    show_logZ=False,
    show_fd=True,
    logZ_func=fermion_logZ_numeric_pimc_predict
)


In [ ]:
from benchmarking import (scaling_model, fit_runtime_scaling,
                          plot_runtime_scaling_fit)

In [ ]:
result = plot_runtime_scaling_fit(logZ_func=fermion_logZ_numeric_pimc_predict,
    
    tau=5.0,
    N=2000,
    d=2,
    n_start=0,
    n_end=5000,
    n_step=250,
    repeats=5
)

In [ ]:
result = plot_runtime_scaling_fit(
    tau=0.1,
    N=2000,
    d=2,
    n_start=0,
    n_end=5000,
    n_step=250,
    repeats=3
)

In [ ]:
from benchmarking import (scaling_model_tau_negpow, highest_shell_zero_temp,
                          fit_runtime_vs_tau_negpow, plot_runtime_vs_tau_negpow_fit)

In [ ]:
result = plot_runtime_vs_tau_negpow_fit(logZ_func=fermion_logZ_numeric_pimc_predict,
    
    tau_start=0.1,
    tau_end=20.1,
    tau_step=0.5,
    N=200,
    n=1000,
    d=2,
    repeats=1
)

In [ ]:
from benchmarking import (scaling_model_N_negpow, highest_shell_zero_temp,
                          fit_runtime_vs_N_negpow, plot_runtime_vs_N_negpow_fit)

In [ ]:
result = plot_runtime_vs_N_negpow_fit(logZ_func=fermion_logZ_numeric_pimc_predict,
    
    N_start=1,
    N_end=30,
    N_step=1,
    tau=1.5,
    n=100,
    d=2,
    repeats=100
)